In [1]:
import pandas
import os
import numpy
import scipy
import matplotlib.pyplot as plt
import mod_gaussQuad
import mod_stefan as analytical_stefan
from mod_stefan import stefan_variables

In [2]:
def return_outputframe_list(dir, filename_append):
    df_list = []
    for idx, i in enumerate(range(100, 901, 100)):
        filename = 't' + str(i) + filename_append
        filepath = os.path.join(dir, filename)
        #print(filepath)
        try:
            df = pandas.read_csv(filepath)
            df_list.append(df)
        except FileNotFoundError:
            print(f"File not found: {filepath}")
        except pandas.errors.EmptyDataError:
            print(f"File is empty: {filepath}")
        except Exception as e:
            print(f"An error occurred while reading {filepath}: {e}")
    return df_list
    


In [3]:
dir = r'D:\thesis\stefan_problem\csv_272_w_short_lineprobe'
filename_append = '_short_lineprobe.csv'

df_list = return_outputframe_list(dir= dir, filename_append= filename_append)

In [23]:
def return_processed_data(outputframe_list):
    temp_threshold = 1e-3
    vol_frac_threshold = 1e-9
    processed_list = []
    for outputframe in outputframe_list:
        filtered_frame = outputframe[
            ~(
                (outputframe['Temperature (K)'].between(272 - temp_threshold, 272 + temp_threshold)) |
                (outputframe['Solid Volume Fraction of h2o'] > (1 - vol_frac_threshold))
            )
        ]
        #HERE TAIL 2 OR TAIL 3?
        extracted_data = filtered_frame[['Temperature (K)', 'Solid Volume Fraction of h2o', 'X (m)']].tail(2)
        extracted_data = extracted_data.to_numpy()
        processed_list.append(extracted_data)
        
    return processed_list

def return_approximate_interface_location(processed_list):
    pos_interface = []
    for data in processed_list:
        xx = data[:][:,-1]

        average_solid_vol_fraction = numpy.mean(data[:][:,1])
        print('x coordinates: ', xx)
        print('average solid volume fraction of water: ', average_solid_vol_fraction)
        average_interface_pos = xx[1] - (xx[1]-xx[0])*average_solid_vol_fraction
        print('approximated position of the interface: ', average_interface_pos)
        pos_interface.append(average_interface_pos)
    return pos_interface
   


In [24]:
processed_list = return_processed_data(outputframe_list= df_list)
#print(processed_list)

In [25]:
num_pos_interface= return_approximate_interface_location(processed_list)

x coordinates:  [0.003 0.004]
average solid volume fraction of water:  0.2989271283149719
approximated position of the interface:  0.003701072871685028
x coordinates:  [0.005 0.006]
average solid volume fraction of water:  0.690666377544403
approximated position of the interface:  0.005309333622455597
x coordinates:  [0.005 0.006]
average solid volume fraction of water:  8.558714988896737e-14
approximated position of the interface:  0.005999999999999914
x coordinates:  [0.007 0.008]
average solid volume fraction of water:  0.7006142735481262
approximated position of the interface:  0.007299385726451874
x coordinates:  [0.007 0.008]
average solid volume fraction of water:  0.206752672791481
approximated position of the interface:  0.0077932473272085195
x coordinates:  [0.009 0.01 ]
average solid volume fraction of water:  0.9169920086860656
approximated position of the interface:  0.009083007991313842
x coordinates:  [0.009 0.01 ]
average solid volume fraction of water:  0.6030848026275

In [26]:
timesteps = numpy.linspace(100, 900, 9)
init_vars = stefan_variables(
    c_p = 4200,
    kappa = 0.6,
    rho = 1000,
    T_l = 300,
    T_m = 273,
    h_m = 333700,
    quadrature= mod_gaussQuad.legendreGaussQuad(ngp= 10, dim= 1)
)

analytical_pos_interface = analytical_stefan.interface_position(vars= init_vars, t_all = timesteps)

d:\thesis\mod_stefan.py:47: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  roots = fsolve(f_lam, lam_init)


In [27]:
[print(abs(an_pos - num_pos)) for an_pos, num_pos in zip(analytical_pos_interface, num_pos_interface)]
#print(analytical_pos_interface)
#print(num_pos_interface)

0.00042575756901399794
0.0006773383003700869
0.00032698748496585474
0.0007487551211098134
0.0004694196626907985
0.0010601567530403727
0.0007312454411805801
0.0005141180420376833
0.001209400242385068


[None, None, None, None, None, None, None, None, None]